## Fetch Duck DB artifacts, load them to Postgres

In [21]:
# Set up .env path
import sys
import os
from dotenv import load_dotenv

load_dotenv()
DB_PATH = os.environ.get("DB_PATH")
if DB_PATH is None:
    raise RuntimeError(
        "DB_PATH not set. Copy .env.example to .env and fill in your local path."
    )

In [22]:
import duckdb

con = duckdb.connect(f"{DB_PATH}marketplace-analytics.duckdb")


In [ ]:
# Export table to a series of chunked CSVs, so you can upload them to Postgres next

con.execute("""
COPY (
    SELECT * FROM staging.stg_events
)
TO 'stg_events_*.csv'
(HEADER, PER_THREAD_OUTPUT TRUE)
""")

## Connect to Postgres

### 1. Create Docker network (will be needed later):

Create a Docker network so Grafana and Postgres Docker containers can communicate (free to chose the name):
```
docker network create [your network]
```

### 2. Set up the postgres db in docker (run in terminal):

Set up Postgres within a Docker container with an accompanying volume, so the data generated in it persists (see docker-compose.yml in this repo for congig details)

Then, from the folder that contains the yaml file, run
``` docker compose up -d ```



In [1]:
# Connect to your Postgres DB in this notebook

import psycopg2
import pandas as pd

conn = psycopg2.connect(
    dbname="analytics",
    user="admin",
    password="alanzin",
    host="localhost",
    port=5432
)

conn.autocommit = True

In [ ]:
# Quick sense check: what DB are you on?

pd.read_sql_query("SELECT current_database();", conn)

In [ ]:
# Create schemas before running upload bash loop

pd.read_sql_query("CREATE SCHEMA staging", conn)
pd.read_sql_query("CREATE SCHEMA marts", conn)

# Quick sense check: were they created correctly?
pd.read_sql_query("SELECT schema_name FROM information_schema.schemata", conn)


In [ ]:
# Create rarget table: stg_events (same as in Duck DB)

pd.read_sql_query("""
                  CREATE TABLE staging.stg_events (
                    user_id BIGINT,
                    user_session VARCHAR,
                    product_id BIGINT,
                    event_time TIMESTAMP,
                    event_type TEXT,
                    event_name TEXT,
                    variant_user TEXT,
                    variant_session TEXT,
                    brand TEXT,
                    category_code VARCHAR
                  )
""", conn)

In [ ]:
# Loop over CSV files to copy them into the new Postgres table
# Notice the file path: it must land on the folder where you store the CSVs

import glob

cur = conn.cursor()

for filepath in glob.glob("stg_events_csvs/data_*.csv"):
    print(f"Loading {filepath}...")
    with open(filepath, "r") as f:
        cur.copy_expert("COPY staging.stg_events FROM STDIN DELIMITER ',' CSV HEADER", f)
    conn.commit()
    print(f"Done: {filepath}")

cur.close()
conn.close()

In [ ]:
# This table will have ~110 million rows; you should index it for faster querying

# In production environments with much larger tables, you'd use more sophisticated
# strategies e.g. partitioning, z-ordering etc.

cur.execute("CREATE INDEX IF NOT EXISTS idx_stg_events_id ON staging.stg_events(event_type, event_time, user_session)")
conn.commit()


In [ ]:
# Quick sense check: count the full table

pd.read_sql_query("""
                  SELECT count(*) FROM staging.stg_events
""", conn)

# Connect Grafana

## 3. Run Grafana container on Docker

```
docker run -d \
  --name grafana \
  --network [your network] \
  -p 3000:3000 \
  grafana/grafana
```

TO DO: ADD PRINTS TO README AND NOTE HERE ABOUT GOING ON IN GRAFANA